In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/annotated/checkpoints/pre_cell_0.pickle

trying: ['tpch_parent']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['pd']


me:  0
trying: ['STORAGE_OPTS']
me:  0
Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable pd
Declaring variable STORAGE_OPTS


In [4]:
%%RecordEvent
%%time
### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = f"{data_folder}/lineitem.tbl"
    # Read CSV on the GPU
    df = pd.read_csv(
        data_path,
        sep="|",
        **storage_options
    )
    print(df.columns)
    # Convert date columns to datetime64[ns] on the GPU
    date_cols = ["L_SHIPDATE", "L_RECEIPTDATE", "L_COMMITDATE"]
    df = df.astype({col: "datetime64[ns]" for col in date_cols})
    return df

lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

Index(['L_ORDERKEY', 'L_PARTKEY', 'L_SUPPKEY', 'L_LINENUMBER', 'L_QUANTITY',
       'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX', 'L_RETURNFLAG',
       'L_LINESTATUS', 'L_SHIPDATE', 'L_COMMITDATE', 'L_RECEIPTDATE',
       'L_SHIPINSTRUCT', 'L_SHIPMODE', 'L_COMMENT', 'L_DUMMY'],
      dtype='object')


CPU times: user 3min 57s, sys: 23.8 s, total: 4min 21s
Wall time: 4min 17s


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/rewritten/cell_26/checkpoints/post_cell_0_try_1.pickle

migration speed (bps): 774998060.9916608
---------------------------
variables to migrate:
load_lineitem 144
STORAGE_OPTS 64
lineitem 48
DATA_ROOT 78
tpch_parent 54
pd 72
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/rewritten/cell_26/checkpoints/post_cell_0_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables ['lineitem']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/opt_cell_exec_info_0_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[0], f)


In [8]:
opt_output = Out.get(4)

In [9]:
lineitem_opt = lineitem
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/annotated/checkpoints/post_cell_0.pickle
assert compare_df(lineitem_opt, lineitem)

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['lineitem']


oops


TypeError: 'dict' object is not callable